<!-- ============================================================ -->
<!-- NOTEBOOK HEADER — MLOps Introductory Course on GCP           -->
<!-- ============================================================ -->

<div style="border-bottom: 3px solid #4285F4; padding-bottom: 12px; margin-bottom: 20px;">

<div style="display: flex; align-items: center; justify-content: space-between;">
  <div>
    <img src="https://www.isae-supaero.fr/wp-content/uploads/2025/03/logo.svg" width="180">
  </div>
  <div style="text-align: right;">
    <img src="https://user-images.githubusercontent.com/63151412/167391313-4683cc69-2bf6-4597-b767-5c18e2bbbfa0.png" width="180">
  </div>
</div>

# Lab 05a — RAG Corpus & Retrieval — ✅ Solution

**Course:** MLOps Introductory Course on GCP · M2 Data Science · ISAE-SUPAERO  
**Lab created by:** Headmind Partners AI & Blockchain  
**Estimated duration:** ~1h00

</div>

## 📋 Lab Overview

**Retrieval-Augmented Generation (RAG)** combines information retrieval with large language models to generate responses grounded in external knowledge. Instead of relying solely on training data, RAG systems dynamically fetch relevant documents to provide accurate, domain-specific answers — a critical pattern for enterprise AI.

In this first part, you will build the **retrieval backbone** of a RAG system using **Vertex AI RAG Engine**, Google Cloud's managed service for document indexing and semantic search.

### Learning Objectives

1. **Initialize** the Vertex AI SDK and configure a GCP project for RAG.
2. **Create a RAG corpus** with a specified embedding model.
3. **Import and chunk** documents from Cloud Storage into the corpus.
4. **Query** the retriever and **tune** parameters (`top_k`, `distance_threshold`).
5. **Analyze** how chunking and retrieval settings affect search quality.

### Notebook Structure

| # | Section | Focus |
|---|---------|-------|
| 0 | Setup | Install dependencies, authenticate, configure project |
| 1 | RAG Corpus Creation | Create a corpus with embedding model configuration |
| 2 | Document Import | Import documents with chunking parameters |
| 3 | Retrieval Testing | Query the corpus, tune retrieval parameters |

### How to Read This Notebook

- **`# TODO`** — Code you need to write. Look for the `######` delimiters.
- **`✏️ Question`** — A conceptual question. Write your answer in the markdown cell below it.
- Cells **without** a TODO are provided — read them, run them, and make sure you understand them.
- Documentation links are provided in 📖 callouts whenever a new API is introduced.

---
## 0 · Setup

### 0.1 Install dependencies

We need the Vertex AI SDK with RAG Engine support.

In [ ]:
%pip install --upgrade google-cloud-aiplatform -q

### 0.2 Authentication

If you are running this notebook **locally** (not on Vertex AI Workbench), run the next cell to authenticate with your GCP credentials:

> 📖 **Docs:** [Install the gcloud CLI](https://cloud.google.com/sdk/docs/install)

In [ ]:
#!gcloud auth application-default login

### 0.3 Imports

In [ ]:
# ── Standard library ──
import time
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

# ── Vertex AI ──
import vertexai
from vertexai import rag
from vertexai.generative_models import GenerativeModel, Tool

print(f"✅ Imports complete")
print(f"Vertex AI SDK version: {vertexai.__version__}")

### 0.4 Configuration

> 📖 **Vertex AI RAG Engine documentation:**
> - [RAG Engine overview](https://cloud.google.com/vertex-ai/generative-ai/docs/rag-engine/rag-overview)
> - [RAG quickstart](https://cloud.google.com/vertex-ai/generative-ai/docs/rag-engine/rag-quickstart)
> - [Supported regions](https://cloud.google.com/vertex-ai/generative-ai/docs/learn/locations)

> ⚠️ **Important:** Verify that your region supports RAG Engine. `europe-west3` is supported.

In [ ]:
# ✅ SOLUTION
PROJECT_ID = "your-project-id"  # @param {type:"string"}
LOCATION = "europe-west3"  # @param {type:"string"}
BUCKET_NAME = "bucket-lab05"  # @param {type:"string"}
YOUR_NAME = "..."  # @param {type:"string"}
EXPERIMENT_ID = "baseline"  # @param {type:"string"}

DISPLAY_NAME = f"rag-corpus-{YOUR_NAME}-{EXPERIMENT_ID}"
PATHS = [f"gs://{BUCKET_NAME}/{YOUR_NAME}/documents/"]

# Initialize Vertex AI SDK
vertexai.init(project=PROJECT_ID, location=LOCATION)

print(f"✅ Vertex AI initialized")
print(f"Project: {PROJECT_ID}")
print(f"Location: {LOCATION}")
print(f"Corpus name: {DISPLAY_NAME}")

---
## 1 · RAG Corpus Creation

A **RAG corpus** is a collection of documents indexed with vector embeddings for semantic search. Vertex AI RAG Engine manages the full pipeline: chunking, embedding, indexing, and serving.

By default, RAG Engine uses **RagManagedDb** as the vector database — no manual setup required.

We'll use Google's **text-embedding-005** model (768 dimensions), the latest version optimized for semantic similarity.

> 📖 **Docs:**
> - [Corpus management](https://cloud.google.com/vertex-ai/generative-ai/docs/rag-engine/manage-your-rag-corpus)
> - [Text embedding models](https://cloud.google.com/vertex-ai/generative-ai/docs/embeddings/get-text-embeddings)
> - [RagManagedDb](https://docs.cloud.google.com/vertex-ai/generative-ai/docs/rag-engine/use-ragmanageddb-with-rag)

In [ ]:
# ✅ SOLUTION
# Configure the embedding model
embedding_model_config = rag.RagEmbeddingModelConfig(
    vertex_prediction_endpoint=rag.VertexPredictionEndpoint(
        publisher_model="publishers/google/models/text-embedding-005"
    )
)

# Create the RAG corpus (uses RagManagedDb by default)
rag_corpus = rag.create_corpus(
    display_name=DISPLAY_NAME,
    backend_config=rag.RagVectorDbConfig(
        rag_embedding_model_config=embedding_model_config
    ),
)

print(f"✅ RAG corpus created")
print(f"Corpus resource name: {rag_corpus.name}")
print(f"Embedding model: text-embedding-005 (768 dimensions)")

**✏️ Question 1 — Embedding Models**

a) Name **two other embedding models** (from any provider) that could be used in a RAG system.  
b) What factors would you consider when choosing an embedding model for a **multilingual** RAG application?

---
*✅ Solution:*

a) Examples include: **OpenAI text-embedding-3-large** (3072 dims), **Cohere embed-multilingual-v3**, **Sentence-BERT all-MiniLM-L6-v2** (384 dims, open-source), **BGE-M3** (BAAI, multilingual, open-source).

b) Key factors for multilingual RAG: **language coverage** (does the model support your target languages?), **MTEB/MIRACL benchmark scores** for cross-lingual retrieval, **dimensionality** (higher = more expressive but costlier), **cost** (API pricing vs. self-hosting), **context length** (max input tokens), and **domain fit** (general vs. specialized).

---

---
## 2 · Document Import

### 2.1 Prerequisites

Before importing, make sure you have run the **`pdf_to_bucket.ipynb`** utility notebook to convert your PDFs to `.txt` and upload them to Cloud Storage.

> 💡 **Tip:** Vertex AI RAG Engine now supports direct PDF import, but for this lab we use `.txt` files to understand the full preprocessing pipeline.

### 2.2 Import files into corpus

The key chunking parameters are:

- **`chunk_size`**: Maximum tokens per chunk (typical: 512–1024). Smaller = more precise retrieval but less context per chunk.
- **`chunk_overlap`**: Tokens shared between consecutive chunks. Prevents information loss at chunk boundaries.
- **`max_embedding_requests_per_min`**: Rate limit to avoid quota errors.

> 📖 **Docs:**
> - [`rag.import_files()`](https://docs.cloud.google.com/vertex-ai/generative-ai/docs/samples/generativeaionvertexai-rag-import-files?hl=en)
> - [Chunking strategies](https://docs.cloud.google.com/vertex-ai/generative-ai/docs/rag-engine/fine-tune-rag-transformations)

In [ ]:
# ✅ SOLUTION
response = rag.import_files(
    rag_corpus.name,
    PATHS,
    transformation_config=rag.TransformationConfig(
        chunking_config=rag.ChunkingConfig(
            chunk_size=512,
            chunk_overlap=100,
        ),
    ),
    max_embedding_requests_per_min=200,
)

print(f"✅ Documents imported to corpus")
print(f"Response: {response}")

> 💡 **Tip:** The import is asynchronous. For large document sets, check progress in the **Vertex AI Console → RAG Engine**.

**✏️ Question 2 — Chunking Strategy**

a) What is **chunk overlap** and why is it necessary in RAG systems?  
b) What problems might arise if you set `chunk_overlap=0`?  
c) A colleague suggests using `chunk_size=2048` with `chunk_overlap=0`. In what scenario could this be a reasonable choice?

---
*✅ Solution:*

a) **Chunk overlap** means consecutive chunks share tokens at their boundaries. For example, with chunk_size=512 and overlap=100, each chunk starts 412 tokens after the previous one. It prevents key information from being split across chunks, maintains context continuity, and improves retrieval recall near boundaries.

b) With `chunk_overlap=0`: sentences may be cut arbitrarily, references (pronouns, "this method"…) lose context, and queries matching text near boundaries may not find a complete answer in any single chunk.

c) Large chunks with no overlap can work when documents are **already well-structured** with clear section boundaries (e.g., a FAQ database where each entry is self-contained), or when storage/cost constraints are significant and a slight recall loss is acceptable.

---

---
## 3 · Retrieval Testing

Before connecting retrieval to a language model, it's important to test the retriever **in isolation**. This helps tune parameters without the added complexity of LLM generation.

Key parameters:
- **`top_k`**: Number of most similar chunks to return (typical: 3–10).
- **`vector_distance_threshold`**: Only return chunks closer than this threshold (0–1, lower = more similar).

> 📖 **Docs:**
> - [`RagRetrievalConfig`](https://docs.cloud.google.com/php/docs/reference/cloud-ai-platform/latest/V1.RagRetrievalConfig)
> - [`rag.retrieval_query()`](https://docs.cloud.google.com/vertex-ai/generative-ai/docs/samples/generativeaionvertexai-rag-retrieval-query?hl=en)
> - [More documentation :)](https://deepwiki.com/googleapis/python-aiplatform/10-development-and-testing#retrieval-and-search-configuration)

In [ ]:
# ✅ SOLUTION
rag_retrieval_config = rag.RagRetrievalConfig(
    top_k=5,
    filter=rag.Filter(vector_distance_threshold=0.5),
)

response = rag.retrieval_query(
    rag_resources=[
        rag.RagResource(rag_corpus=rag_corpus.name)
    ],
    text="Who is Cosette?",
    rag_retrieval_config=rag_retrieval_config,
)

print(f"✅ Retrieved {len(response.contexts.contexts)} chunks\n")
for i, ctx in enumerate(response.contexts.contexts, 1):
    print(f"─── Chunk {i} (cosine distance: {ctx.score:.3f}) ───")
    print(ctx.text[:200] + "...\n")

> 💡 **Experimentation tips:**
> - Try queries with different specificity levels (broad vs. narrow)
> - Ask an **out-of-context** question (e.g., "What is the capital of France?") and observe the distances
> - Compare results with different `top_k` and `distance_threshold` values

**✏️ Question 3 — Retrieval Tuning**

a) You query "What is reinforcement learning?" against a corpus about the DeepSeek-R1 paper. The top 5 chunks all have distance > 0.7. What does this tell you, and how should you adjust the threshold?  
b) What is the role of the **distance threshold** in production RAG systems? What happens if it is too low? Too high?  
c) Why is it useful to test the retriever **separately** before connecting it to the LLM?

---
*✅ Solution:*

a) Distances > 0.7 indicate **low semantic similarity** — the retrieved chunks are not very relevant. You should **lower** the threshold (e.g., 0.5) to filter out these irrelevant results. The query may also be too broad for the corpus content.

b) The distance threshold acts as a **quality gate**: it filters out chunks that are too dissimilar to the query. If set **too low** (strict), you may filter out relevant chunks and return empty results. If set **too high** (permissive), irrelevant chunks pollute the context and can cause hallucination in the LLM response.

c) Testing the retriever separately lets you: diagnose whether poor answers come from bad retrieval or bad generation, tune retrieval parameters without LLM cost, and verify that the right documents are being found before adding LLM complexity.

---

### 3.1 Save corpus reference for Lab 05b

Copy the corpus resource name below — you will need it in the next notebook.

In [ ]:
print(f"\n📋 Copy this corpus resource name for Lab 05b:")
print(f"   {rag_corpus.name}")

---
## Summary

In this lab, you learned to:

| Step | What you did | Tool / Feature used |
|------|-------------|---------------------|
| **Setup** | Configured GCP project and initialized Vertex AI | `vertexai.init()` |
| **Corpus creation** | Created a RAG corpus with text-embedding-005 | `rag.create_corpus()`, `RagEmbeddingModelConfig` |
| **Document import** | Imported chunked documents from Cloud Storage | `rag.import_files()`, `ChunkingConfig` |
| **Retrieval testing** | Queried the retriever and tuned parameters | `rag.retrieval_query()`, `RagRetrievalConfig` |

**Next lab:** In **Lab 05b**, you will connect this retriever to a Gemini model for grounded generation and evaluate the full RAG system using RAGAS metrics.